# HR Candidate Profile Parser (LangChain StructuredOutputParser)


In [ ]:
!pip install -U langchain langchain-community langchain-core transformers==4.52.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 123.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 115.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 87.3 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.31.0
    Uninstalling huggingface_h

In [ ]:
!pip install -U langchain_openai openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 77.8 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.54.0
    Uninstalling openai-2.54.0:
      Successfully uninstalled openai-2.54.0


In [ ]:
!pip install -U langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 11.1 MB/s eta 0:00:00


In [ ]:
import os
import json
import torch
import re
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import PromptTemplate
from langchain_classic.output_parsers import OutputFixingParser
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

## API Key

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret("GROQ_API_KEY")
except Exception:
    api_key = None

if not api_key:
    api_key = input("Enter your Groq API key: ").strip()

os.environ["GROQ_API_KEY"] = api_key

## 1. Define the Output Schema

In [ ]:
response_schemas = [
    ResponseSchema(
        name="full_name",
        description="The candidate's full name as a string",
    ),
    ResponseSchema(
        name="email",
        description="The candidate's email address as a string",
    ),
    ResponseSchema(
        name="education",
        description=(
            "A list of education entries. Each entry is a JSON object with keys: "
            "'degree' (string), 'institution' (string), 'year' (integer)."
        ),
    ),
    ResponseSchema(
        name="skills",
        description="A list of strings representing the candidate's skills",
    ),
    ResponseSchema(
        name="experience",
        description=(
            "A list of experience entries. Each entry is a JSON object with keys: "
            "'role' (string), 'company' (string), 'years' (string, e.g. '2020-2023' or '2023-Present')."
        ),
    ),
]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = output_parser.get_format_instructions()
print(format_instructions)


The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"full_name": string  // The candidate's full name as a string
	"email": string  // The candidate's email address as a string
	"education": string  // A list of education entries. Each entry is a JSON object with keys: 'degree' (string), 'institution' (string), 'year' (integer).
	"skills": string  // A list of strings representing the candidate's skills
	"experience": string  // A list of experience entries. Each entry is a JSON object with keys: 'role' (string), 'company' (string), 'years' (string, e.g. '2020-2023' or '2023-Present').
}
```


## 2. Build the Prompt Template


In [ ]:
template = """You are an expert HR assistant that extracts structured information from resume text.

Read the resume snippet below carefully and extract the candidate's information.
Return ONLY the data in the exact format specified below, with no extra commentary.

Resume Text:
{resume_text}

{format_instructions}
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["resume_text"],
    partial_variables={"format_instructions": format_instructions},
)


## 3. Input


In [ ]:
%pip install -q -U pypdf python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 29.8 MB/s eta 0:00:00


In [ ]:
# !find /kaggle/input -type f

/kaggle/input/datasets/basmalaaa2/basmala-cv/Basmala_AboElhamed_CV.pdf


In [ ]:
import os

cv_path = "/content/Basmala_AboElhamed_CV.pdf"

def extract_text_from_cv(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".pdf":
        from pypdf import PdfReader
        reader = PdfReader(path)
        return "\n".join(page.extract_text() or "" for page in reader.pages)
    elif ext == ".docx":
        from docx import Document
        doc = Document(path)
        return "\n".join(p.text for p in doc.paragraphs)
    elif ext == ".txt":
        with open(path, "r", encoding="utf-8") as f:
            return f.read()
    else:
        raise ValueError(f"Unsupported file type: {ext}")

resume_text = extract_text_from_cv(cv_path)
print(resume_text)

BASMALA ABOELHAMED 
•  basmalaaboelhamed@gmail.com  • +20 111 072 1378 
linkedin.com/in/basmala-aboelhamed-7338b42a7  •  basmalaaboelhamed.github.io/portfolio 
PROFESSIONAL SUMMARY 
Computer Science student specializing in Machine Learning, Deep Learning, and Data Analytics, with hands -on experience building end-
to-end AI systems across Computer Vision, NLP, and Retrieval-Augmented Generation (RAG). Skilled in Python, TensorFlow, scikit-learn, 
LangChain, and SQL, with proven experience developing CNN-based detection models, RAG-powered research and customer-support 
assistants, and full data analysis pipelines from raw data to interactive dashboards. Strong leadership, mentorship, and t eamwork skills 
built as a Mentor and Operations Team member within the ICPC Community, with competitive problem-solving experience from 
participating in the ECPC Qualifications. Recognized top graduate with industry certifications from NVIDIA, ITI, ITIDA, and Orange Digital 
Center; a fast learner 

## 4. Run LLM and Parse the Output

In [ ]:
import os
print(bool(os.environ.get("GROQ_API_KEY")))

True


In [ ]:
import requests

url = "https://api.groq.com/openai/v1/models"
headers = {"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"}
response = requests.get(url, headers=headers)
for m in response.json().get("data", []):
    print(m["id"])

meta-llama/llama-prompt-guard-2-22m
canopylabs/orpheus-v1-english
allam-2-7b
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3
qwen/qwen3.8-27b
openai/gpt-oss-20b
whisper-large-v3-turbo
openai/gpt-oss-120b
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-arabic-saudi


In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

fixing_parser = OutputFixingParser.from_llm(parser=output_parser, llm=llm)

_input = prompt.format_prompt(resume_text=resume_text)
raw_output = llm.invoke(_input.to_string())

try:
    parsed_result = output_parser.parse(raw_output.content)
except Exception:
    parsed_result = fixing_parser.parse(raw_output.content)

for key in ["education", "skills", "experience"]:
    if isinstance(parsed_result.get(key), str):
        parsed_result[key] = json.loads(parsed_result[key])

print(json.dumps(parsed_result, indent=2, ensure_ascii=False))

{
  "full_name": "BASMALA ABOELHAMED",
  "email": "basmalaaboelhamed@gmail.com",
  "education": [
    {
      "degree": "Bachelor of Computing and Information System",
      "institution": "Arab Academy for Science, Technology and Maritime Transport (AASTMT)",
      "year": 2027
    }
  ],
  "skills": [
    "Python",
    "Java",
    "C++",
    "SQL",
    "JavaScript",
    "HTML/CSS",
    "TensorFlow",
    "Keras",
    "scikit-learn",
    "PyTorch",
    "pandas",
    "NumPy",
    "XGBoost",
    "Matplotlib",
    "LangChain",
    "RAG",
    "LLMs",
    "Ollama",
    "Prompt Engineering",
    "Sentence Transformers",
    "Semantic Search",
    "ChromaDB",
    "FAISS",
    "FastAPI",
    "Streamlit",
    "Google Gemini API",
    "REST APIs",
    "MySQL",
    "MS SQL Server",
    "PostgreSQL",
    "Docker",
    "Git",
    "GitHub",
    "JavaFX",
    "Android Studio",
    "VS Code",
    "PyCharm",
    "IntelliJ",
    "NetBeans",
    "Jupyter",
    "Anaconda",
    "Google Colab",
    "Arabic"

## 5. Streamlit GUI + ngrok

In [ ]:
!pip install -q -U streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 117.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 122.1 MB/s eta 0:00:00


In [ ]:
%%writefile app.py
import html
import io
import json
import os

import pandas as pd
import streamlit as st
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq

try:
    from langchain_classic.output_parsers import (
        OutputFixingParser,
        ResponseSchema,
        StructuredOutputParser,
    )
except ImportError:
    from langchain.output_parsers import (
        OutputFixingParser,
        ResponseSchema,
        StructuredOutputParser,
    )

MAX_CHARS = 24000
DEFAULT_MODEL = "openai/gpt-oss-20b"

# ----------------------------------------------------------------------------
# 1. Output schema + prompt
# ----------------------------------------------------------------------------
response_schemas = [
    ResponseSchema(
        name="full_name",
        description="The candidate's full name as a string",
    ),
    ResponseSchema(
        name="email",
        description="The candidate's email address as a string",
    ),
    ResponseSchema(
        name="education",
        description=(
            "A list of education entries. Each entry is a JSON object with keys: "
            "'degree' (string), 'institution' (string), 'year' (integer)."
        ),
    ),
    ResponseSchema(
        name="skills",
        description="A list of strings representing the candidate's skills",
    ),
    ResponseSchema(
        name="experience",
        description=(
            "A list of experience entries. Each entry is a JSON object with keys: "
            "'role' (string), 'company' (string), 'years' (string, e.g. '2020-2023' or '2023-Present')."
        ),
    ),
]

output_parser = StructuredOutputParser.from_response_schemas(response_schemas)

TEMPLATE = """You are an expert HR assistant that extracts structured information from resume text.

Read the resume snippet below carefully and extract the candidate's information.
Return ONLY the data in the exact format specified below, with no extra commentary.

Resume Text:
{resume_text}

{format_instructions}
"""

prompt = PromptTemplate(
    template=TEMPLATE,
    input_variables=["resume_text"],
    partial_variables={"format_instructions": output_parser.get_format_instructions()},
)


# ----------------------------------------------------------------------------
# 2. Helpers
# ----------------------------------------------------------------------------
def extract_text_from_upload(uploaded_file) -> str:
    name = uploaded_file.name.lower()
    data = uploaded_file.getvalue()

    if name.endswith(".pdf"):
        from pypdf import PdfReader

        reader = PdfReader(io.BytesIO(data))
        return "\n".join(page.extract_text() or "" for page in reader.pages)

    if name.endswith(".docx"):
        from docx import Document

        doc = Document(io.BytesIO(data))
        parts = [p.text for p in doc.paragraphs]
        for table in doc.tables:
            for row in table.rows:
                parts.append(" | ".join(cell.text for cell in row.cells))
        return "\n".join(parts)

    if name.endswith(".txt"):
        return data.decode("utf-8", errors="ignore")

    raise ValueError("Unsupported file type. Please upload a PDF, DOCX or TXT file.")


def _as_list(value):
    if isinstance(value, str):
        try:
            value = json.loads(value)
        except Exception:
            return [value] if value.strip() else []
    if value is None:
        return []
    return value if isinstance(value, list) else [value]


def parse_resume(resume_text: str, api_key: str, model: str, temperature: float) -> dict:
    llm = ChatGroq(model=model, temperature=temperature, api_key=api_key)
    fixing_parser = OutputFixingParser.from_llm(parser=output_parser, llm=llm)

    raw_output = llm.invoke(prompt.format_prompt(resume_text=resume_text).to_string())
    try:
        result = output_parser.parse(raw_output.content)
    except Exception:
        result = fixing_parser.parse(raw_output.content)

    for key in ("education", "skills", "experience"):
        result[key] = _as_list(result.get(key))
    return result


def render_skills(skills):
    chips = "".join(
        f"<span style='display:inline-block;margin:3px 6px 3px 0;padding:4px 12px;"
        f"border-radius:14px;background:rgba(99,102,241,.15);border:1px solid rgba(99,102,241,.4);"
        f"font-size:0.9rem'>{html.escape(str(s))}</span>"
        for s in skills
    )
    st.markdown(chips or "_No skills found._", unsafe_allow_html=True)


# ----------------------------------------------------------------------------
# 3. UI
# ----------------------------------------------------------------------------
st.set_page_config(page_title="HR Candidate Profile Parser", page_icon="📄", layout="wide")
st.title("📄 HR Candidate Profile Parser")
st.caption("Upload a CV (PDF / DOCX / TXT) or paste its text, and get a structured candidate profile.")

env_key = os.environ.get("GROQ_API_KEY", "")

with st.sidebar:
    st.header("Settings")
    if env_key:
        api_key = env_key
        st.success("Groq API key loaded from the server.")
    else:
        api_key = st.text_input("Groq API key", type="password", help="Get a free key from console.groq.com")
    model = st.text_input("Model", value=DEFAULT_MODEL)
    temperature = st.slider("Temperature", 0.0, 1.0, 0.0, 0.1)

tab_upload, tab_paste = st.tabs(["Upload CV", "Paste text"])
with tab_upload:
    uploaded = st.file_uploader("CV file", type=["pdf", "docx", "txt"])
with tab_paste:
    pasted = st.text_area("Resume text", height=250)

if st.button("Parse profile", type="primary"):
    try:
        if uploaded is not None:
            resume_text = extract_text_from_upload(uploaded)
        else:
            resume_text = pasted
        resume_text = (resume_text or "").strip()

        if not api_key:
            st.error("Please enter your Groq API key in the sidebar.")
        elif not resume_text:
            st.error("No text found. Upload a text-based CV or paste the resume text. (Scanned PDFs are not supported.)")
        else:
            if len(resume_text) > MAX_CHARS:
                st.warning(f"The CV is long, only the first {MAX_CHARS:,} characters will be used.")
                resume_text = resume_text[:MAX_CHARS]
            with st.spinner("Analyzing the CV..."):
                st.session_state["result"] = parse_resume(resume_text, api_key, model.strip(), temperature)
                st.session_state["resume_text"] = resume_text
    except Exception as exc:
        st.session_state.pop("result", None)
        st.error(f"Something went wrong: {exc}")

result = st.session_state.get("result")
if result:
    st.divider()
    col1, col2 = st.columns(2)
    col1.metric("Full name", result.get("full_name") or "-")
    col2.metric("Email", result.get("email") or "-")

    st.subheader("🎓 Education")
    if result["education"]:
        st.dataframe(pd.DataFrame(result["education"]), hide_index=True)
    else:
        st.write("No education entries found.")

    st.subheader("🛠️ Skills")
    render_skills(result["skills"])

    st.subheader("💼 Experience")
    if result["experience"]:
        st.dataframe(pd.DataFrame(result["experience"]), hide_index=True)
    else:
        st.write("No experience entries found.")

    st.divider()
    json_str = json.dumps(result, indent=2, ensure_ascii=False)
    st.download_button("⬇️ Download JSON", json_str, file_name="candidate_profile.json", mime="application/json")
    with st.expander("Raw JSON"):
        st.code(json_str, language="json")
    with st.expander("Extracted CV text"):
        st.text(st.session_state.get("resume_text", ""))


Overwriting app.py


In [ ]:
import getpass
import subprocess
import sys
import time

from pyngrok import ngrok

PORT = 8501

try:
    from kaggle_secrets import UserSecretsClient
    ngrok_token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
except Exception:
    ngrok_token = os.environ.get("NGROK_AUTHTOKEN")

if not ngrok_token:
    ngrok_token = getpass.getpass("Enter your ngrok authtoken: ").strip()

subprocess.run(["pkill", "-f", "streamlit run"], check=False)
ngrok.kill()
ngrok.set_auth_token(ngrok_token)

proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.port", str(PORT),
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
    ],
    env=os.environ.copy(),
)
time.sleep(6)

tunnel = ngrok.connect(PORT, "http")
print("=" * 60)
print("App is live. Share this link:", tunnel.public_url)
print("=" * 60)
print("Keep this cell running. Interrupt it to shut the app down.")

try:
    proc.wait()
except KeyboardInterrupt:
    pass
finally:
    proc.terminate()
    ngrok.kill()


Enter your ngrok authtoken: ··········
App is live. Share this link: https://delicacy-pronto-suffrage.ngrok-free.dev
Keep this cell running. Interrupt it to shut the app down.
